In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="kX0RjUlt2tVQzkpKm4ZV")
project = rf.workspace("hemavardhan").project("landmines-detection-dataset-sewvx")
version = project.version(1)
dataset = version.download("coco")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 144.9 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.12.0.88
    Uninstalling opencv-python-headless-4.12.0.88:
      Successfully uninstalled opencv-python-headless-4.12.0.88
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Landmines-detection-dataset-1 in coco:: 100%|██████████| 1202/1202 [00:00<00:00, 7954.17it/s]


In [ ]:
# Cell B: SSD300 (run this cell separately)
!pip install torch torchvision pycocotools --quiet

import torch, json
from torchvision.datasets import CocoDetection
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision import transforms
from torch.utils.data import DataLoader
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("Device:", device)

# Paths
TRAIN_ROOT = "/content/Landmines-detection-dataset-1/train"
TRAIN_ANN  = "/content/Landmines-detection-dataset-1/train/_annotations.coco.json"
VAL_ROOT   = "/content/Landmines-detection-dataset-1/val"
VAL_ANN    = "/content/Landmines-detection-dataset-1/valid/_annotations.coco.json"

# Use official SSD weights' transforms (normalization, resizing, etc.)
weights = SSD300_VGG16_Weights.DEFAULT
transform = weights.transforms()  # THIS is important to avoid NaNs

def collate_fn(batch):
    images, targets = [], []
    for img, ann in batch:
        # transform returns tensor already for SSD weights transforms
        img_t = transform(img) if not isinstance(img, torch.Tensor) else img
        images.append(img_t)
        boxes, labels = [], []
        for obj in ann:
            x,y,w,h = obj["bbox"]
            boxes.append([x,y,x+w,y+h])
            labels.append(obj["category_id"])
        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        targets.append({"boxes": boxes, "labels": labels})
    return images, targets

train_dataset = CocoDetection(root=TRAIN_ROOT, annFile=TRAIN_ANN, transform=lambda img: img)  # transform applied in collate
val_dataset   = CocoDetection(root=VAL_ROOT, annFile=VAL_ANN, transform=lambda img: img)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

# Model
model = ssd300_vgg16(weights=weights)
num_classes = 2
# set num_classes correctly (rebuild heads if necessary handled internally in torchvision)
model.head.classification_head.num_classes = num_classes
model.to(device)

# optimizer & AMP
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.001, momentum=0.9, weight_decay=0.0005)
scaler = torch.amp.GradScaler()

num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu"):
            loss_dict = model(images, targets)
            loss = sum(loss for loss in loss_dict.values())
        if torch.isnan(loss):
            print("NaN detected — skipping batch")
            continue
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    print(f"[SSD] Epoch {epoch+1}/{num_epochs}  Loss: {epoch_loss:.4f}")

torch.save(model.state_dict(), "ssd_landmine_best.pth")
print("Saved ssd_landmine_best.pth")

# Evaluation (sequential inference to get image ids)
def evaluate_ssd(model, val_dataset, val_ann, out_json="ssd_preds.json"):
    model.eval()
    coco_gt = COCO(val_ann)
    preds = []
    with torch.no_grad():
        for idx in range(len(val_dataset)):
            img, ann = val_dataset[idx]
            img_t = transform(img).to(device) if not isinstance(img, torch.Tensor) else img.to(device)
            out = model([img_t])[0]
            image_id = val_dataset.ids[idx] if hasattr(val_dataset, "ids") else coco_gt.getImgIds()[idx]
            boxes = out["boxes"].cpu().numpy()
            scores = out["scores"].cpu().numpy()
            labels = out["labels"].cpu().numpy()
            for box, score, label in zip(boxes, scores, labels):
                x1,y1,x2,y2 = box
                preds.append({
                    "image_id": image_id,
                    "category_id": int(label),
                    "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                    "score": float(score)
                })
    with open(out_json, "w") as f:
        json.dump(preds, f)
    coco_dt = coco_gt.loadRes(out_json)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

print("Evaluating SSD ...")
evaluate_ssd(model, val_dataset, VAL_ANN, out_json="ssd_preds.json")

Device: cuda
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
Train size: 836, Val size: 240
[SSD] Epoch 1/50  Loss: 1042.2749
[SSD] Epoch 2/50  Loss: 647.2067
[SSD] Epoch 3/50  Loss: 523.8653
[SSD] Epoch 4/50  Loss: 446.3619
[SSD] Epoch 5/50  Loss: 354.0825
[SSD] Epoch 6/50  Loss: 304.6913
[SSD] Epoch 7/50  Loss: 253.3228
[SSD] Epoch 8/50  Loss: 232.2338
[SSD] Epoch 9/50  Loss: 188.0776
[SSD] Epoch 10/50  Loss: 171.6869
[SSD] Epoch 11/50  Loss: 138.1374
[SSD] Epoch 12/50  Loss: 120.9639
[SSD] Epoch 13/50  Loss: 111.3206
[SSD] Epoch 14/50  Loss: 93.0478
[SSD] Epoch 15/50  Loss: 96.4394
[SSD] Epoch 16/50  Loss: 79.8215
[SSD] Epoch 17/50  Loss: 70.0051
[SSD] Epoch 18/50  Loss: 62.6758
[SSD] Epoch 19/50  Loss: 55.4274
[SSD] Epoch 20/50  Loss: 47.8627
[SSD] Epoch 21/50  Loss: 43.6265
[SSD] Epoch 22/50  Loss: 35.2473
[SSD] Epoch 23/50  Loss: 30.7371
[SSD] Epoch 24/50  Loss: 2

FileNotFoundError: [Errno 2] No such file or directory: '/content/Landmines-detection-dataset-1/val/539_Zone_9_Mine_10_cm_depth__3-9m_TemperatureInCenter__27-C_jpg.rf.ee6d00d412a149d25016a5894c57d411.jpg'